In [1]:
from __future__ import annotations

import logging
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from scipy import stats
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rule_backtest.signals import SIGNAL_CLASSES, BaseSignal
from rule_backtest.data_loader import (
    load_cached_trades,
    merge_aggtrades,
    build_bars,
    compute_factor,
    estimate_avg_bar_seconds,
    RAW_FACTOR_REGISTRY,
    MERGED_FACTOR_REGISTRY,
)

logger = logging.getLogger(__name__)

In [2]:
cache_dir = Path("/Volumes/Lexar/mean_reversion_data/cache")

In [4]:
symbol='SOL/USDC'
cache_subdir: str = "trades_perp"
sym_dir = cache_dir / symbol.replace("/", "_") / cache_subdir
if not sym_dir.exists():
    raise FileNotFoundError(f"Cache directory not found: {sym_dir}")

start_date = '2026-02-02'
end_date = '2026-03-04'
files = sorted(f for f in sym_dir.glob("*.parquet") if not f.name.startswith("._") and start_date <= f.stem <= end_date)

if not files:
    raise FileNotFoundError(f"No parquet files in {sym_dir}")

logger.info("Loading %d cached files from %s", len(files), sym_dir)
dfs = [pd.read_parquet(f) for f in files]
trades = pd.concat(dfs, ignore_index=True)
trades["timestamp"] = pd.to_datetime(trades["timestamp"])
trades.sort_values("timestamp", inplace=True)
trades.reset_index(drop=True, inplace=True)

# Unified column names
trades.rename(columns={"amount": "volume", "cost": "value"}, inplace=True)

logger.info("Loaded %d trades, %s → %s",
            len(trades), trades["timestamp"].iloc[0], trades["timestamp"].iloc[-1])

In [6]:
trades_df = merge_aggtrades(trades)
trades_df.head()

,timestamp,volume,side,value,n_fills,n_raw_trades,first_price,last_price,first_trade_id,last_trade_id,price,price_range
0,2026-02-02 16:00:01.690000+00:00,16.45,sell,1720.3798,2,7,104.59,104.58,304094557,304094563,104.582359,0.01
1,2026-02-02 16:00:01.843000+00:00,40.10,buy,4193.8513,2,3,104.58,104.59,304094564,304094566,104.584820,0.01
2,2026-02-02 16:00:01.970000+00:00,62.43,sell,6528.9294,1,3,104.58,104.58,304094567,304094569,104.580000,0.00
3,2026-02-02 16:00:02.100000+00:00,5.02,sell,525.0418,1,3,104.59,104.59,304094570,304094572,104.590000,0.00
4,2026-02-02 16:00:02.360000+00:00,0.06,buy,6.2754,1,1,104.59,104.59,304094573,304094573,104.590000,0.00


In [7]:
bar_mode = "trade_count"
bars = build_bars(trades_df, "merged", bar_mode, 500)
bar_sec = int(estimate_avg_bar_seconds(bars))
logger.info("Built %d trade-count bars (%d trades/bar, ~%ds avg)",
            len(bars), 500, bar_sec)

## helper functions

In [8]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from datetime import datetime

def calculate_trading_metrics(bars, position_col, returns_col, fee_rate, factor_name, fw, ema_w):
    """
    计算全面的交易绩效指标（修正版：正确处理反向交易）
    """
    
    # 基础数据准备
    df = bars.copy()
    
    # 计算原始收益（不含费用）
    df['raw_returns'] = df[position_col].shift(1) * df['returns']
    
    # 识别持仓变化
    df['position_prev'] = df[position_col].shift(1).fillna(0)
    df['position_change'] = df[position_col] - df['position_prev']
    
    # 初始化信号列
    df['open_signal'] = False
    df['close_signal'] = False
    
    # 识别开仓和平仓时刻（正确处理反向交易）
    for i in range(1, len(df)):
        change = df['position_change'].iloc[i]
        prev_pos = df['position_prev'].iloc[i]
        curr_pos = df[position_col].iloc[i]
        
        if change == 0:
            continue
        
        if abs(change) == 1:
            # 普通开仓或平仓
            if prev_pos == 0:  # 0 -> ±1，开仓
                df.loc[df.index[i], 'open_signal'] = True
            elif curr_pos == 0:  # ±1 -> 0，平仓
                df.loc[df.index[i], 'close_signal'] = True
                
        elif abs(change) == 2:
            # 反向交易：先平仓，再开仓
            # 平仓发生在 bar i 开盘（使用上一根 bar 的持仓）
            df.loc[df.index[i], 'close_signal'] = True
            # 开仓也发生在 bar i 开盘（新方向）
            df.loc[df.index[i], 'open_signal'] = True
    
    # 构建交易列表
    trades = []
    current_trade = None
    
    for i in range(len(df)):
        # 先处理平仓（如果有）
        if df['close_signal'].iloc[i] and current_trade is not None:
            # 平仓
            current_trade['exit_time'] = df.index[i]
            current_trade['exit_price'] = df['open'].iloc[i]  # 以开盘价平仓
            trades.append(current_trade)
            current_trade = None
        
        # 再处理开仓
        if df['open_signal'].iloc[i]:
            direction = df[position_col].iloc[i]  # 当前持仓方向
            current_trade = {
                'entry_time': df.index[i],
                'entry_price': df['open'].iloc[i],
                'direction': direction,
                'exit_time': None,
                'exit_price': None,
                'return_raw': 0,
                'bars_held': 0
            }
        
        # 更新当前持仓的收益
        if current_trade is not None:
            current_trade['return_raw'] += df['raw_returns'].iloc[i]
            current_trade['bars_held'] += 1
    
    # 处理最后一笔未平仓交易
    if current_trade is not None:
        trades.append(current_trade)
    
    # 转换为DataFrame
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
    
    # 统计指标
    if not trades_df.empty:
        # 基本交易统计
        n_trades_total = len(trades_df)
        long_trades = trades_df[trades_df['direction'] == 1]
        short_trades = trades_df[trades_df['direction'] == -1]
        n_long = len(long_trades)
        n_short = len(short_trades)
        
        # 胜率计算
        win_rate_total = (trades_df['return_raw'] > 0).mean() * 100
        win_rate_long = (long_trades['return_raw'] > 0).mean() * 100 if n_long > 0 else 0
        win_rate_short = (short_trades['return_raw'] > 0).mean() * 100 if n_short > 0 else 0
        
        # 盈亏比相关
        winning_trades = trades_df[trades_df['return_raw'] > 0]
        losing_trades = trades_df[trades_df['return_raw'] < 0]
        
        avg_win = winning_trades['return_raw'].mean() if len(winning_trades) > 0 else 0
        avg_loss = losing_trades['return_raw'].mean() if len(losing_trades) > 0 else 0
        
        profit_factor = abs(avg_win * len(winning_trades) / (avg_loss * len(losing_trades))) if len(losing_trades) > 0 and avg_loss != 0 else float('inf')
        
        # 最大连续亏损
        trades_df['is_win'] = trades_df['return_raw'] > 0
        trades_df['consecutive_losses'] = (~trades_df['is_win']).groupby((trades_df['is_win'] != trades_df['is_win'].shift()).cumsum()).cumsum()
        max_consecutive_losses = trades_df['consecutive_losses'].max()
        
        # 平均持仓bar数
        avg_bars_held = trades_df['bars_held'].mean()
        
        # 最佳/最差交易
        best_trade = trades_df['return_raw'].max()
        worst_trade = trades_df['return_raw'].min()
        expectancy = trades_df['return_raw'].mean()
        
    else:
        n_trades_total = n_long = n_short = 0
        win_rate_total = win_rate_long = win_rate_short = 0
        profit_factor = 0
        max_consecutive_losses = 0
        avg_bars_held = 0
        best_trade = worst_trade = expectancy = 0
        avg_win = 0  # 添加这一行
        avg_loss = 0  # 添加这一行
        long_trades = short_trades = pd.DataFrame()
    
    # 累计收益（不含费用）
    df['cumulative_raw'] = (1 + df['raw_returns']).cumprod()
    df['cumulative_long_raw'] = (1 + df['raw_returns'] * (df[position_col] == 1).astype(int)).cumprod()
    df['cumulative_short_raw'] = (1 + df['raw_returns'] * (df[position_col] == -1).astype(int)).cumprod()
    
    # 累计收益（含费用）
    df['cumulative_net'] = (1 + df[returns_col]).cumprod()
    
    # 风险指标
    risk_free_rate = 0  # 0%年化无风险利率，分钟级数据
    
    # 夏普比率
    excess_returns = df[returns_col] - risk_free_rate
    sharpe_ratio = np.sqrt(len(df)) * excess_returns.mean() / excess_returns.std() if excess_returns.std() > 0 else 0
    
    # 最大回撤
    rolling_max = df['cumulative_net'].expanding().max()
    drawdown = (df['cumulative_net'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    max_drawdown_duration = (drawdown == 0).astype(int).groupby(drawdown.ne(0).astype(int).cumsum()).cumsum().max()
    
    # 卡尔玛比率
    calmar_ratio = (df['cumulative_net'].iloc[-1] - 1) / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # 费用统计
    total_fees = (fee_rate * df['position_change'].abs()).sum()
    avg_fee_per_trade = total_fees / n_trades_total if n_trades_total > 0 else 0
    
    # 反向交易统计（用于诊断）
    n_reversals = (df['position_change'].abs() == 2).sum()
    
    # 年化指标（假设分钟级数据）
    bars_per_year = 365 * 24 * 60
    total_return_net = df['cumulative_net'].iloc[-1] - 1
    annualized_return = (1 + total_return_net) ** (bars_per_year / len(df)) - 1
    annualized_volatility = df[returns_col].std() * np.sqrt(bars_per_year)
    
    # 收集所有指标
    metrics = {
        # 基本信息
        'factor_name': factor_name,
        'fw': fw,
        'ema_w': ema_w,
        'start_date': df.index[0],
        'end_date': df.index[-1],
        'total_bars': len(df),
        
        # 收益指标
        'total_return_net': total_return_net,
        'total_return_long_raw': df['cumulative_long_raw'].iloc[-1] - 1,
        'total_return_short_raw': df['cumulative_short_raw'].iloc[-1] - 1,
        'benchmark_return': df['benchmark_cumulative'].iloc[-1] - 1,
        'annualized_return': annualized_return,
        'annualized_volatility': annualized_volatility,
        
        # 交易统计
        'n_trades_total': n_trades_total,
        'n_long': n_long,
        'n_short': n_short,
        'n_reversals': n_reversals,
        'avg_bars_held': avg_bars_held,
        
        # 胜率
        'win_rate_total': win_rate_total,
        'win_rate_long': win_rate_long,
        'win_rate_short': win_rate_short,
        
        # 盈亏分析
        'profit_factor': profit_factor,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'expectancy': expectancy,
        'best_trade': best_trade,
        'worst_trade': worst_trade,
        'max_consecutive_losses': max_consecutive_losses,
        
        # 风险指标
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'max_drawdown_duration': max_drawdown_duration,
        'calmar_ratio': calmar_ratio,
        
        # 费用统计
        'total_fees': total_fees,
        'avg_fee_per_trade': avg_fee_per_trade,
        'fee_rate': fee_rate,
    }
    
    return metrics, trades_df, df


def print_metrics_summary(metrics):
    """打印指标汇总"""
    print("=" * 60)
    print(f"策略绩效报告 - {metrics['factor_name']} (fw={metrics['fw']}, ema={metrics['ema_w']})")
    print("=" * 60)
    print(f"回测期间: {metrics['start_date']} 至 {metrics['end_date']}")
    print(f"总Bar数: {metrics['total_bars']}")
    print("-" * 60)
    print("【收益指标】")
    print(f"  策略总收益 (净): {metrics['total_return_net']:.2%}")
    print(f"  多头收益 (原始): {metrics['total_return_long_raw']:.2%}")
    print(f"  空头收益 (原始): {metrics['total_return_short_raw']:.2%}")
    print(f"  基准收益: {metrics['benchmark_return']:.2%}")
    print(f"  年化收益率: {metrics['annualized_return']:.2%}")
    print("-" * 60)
    print("【交易统计】")
    print(f"  总交易次数: {metrics['n_trades_total']}")
    print(f"  多头交易: {metrics['n_long']}")
    print(f"  空头交易: {metrics['n_short']}")
    print(f"  反向交易: {metrics['n_reversals']}")
    print(f"  平均持仓Bar数: {metrics['avg_bars_held']:.2f}")
    print("-" * 60)
    print("【胜率统计】")
    print(f"  总胜率: {metrics['win_rate_total']:.2f}%")
    print(f"  多头胜率: {metrics['win_rate_long']:.2f}%")
    print(f"  空头胜率: {metrics['win_rate_short']:.2f}%")
    print("-" * 60)
    print("【盈亏分析】")
    print(f"  盈亏比 (Profit Factor): {metrics['profit_factor']:.3f}")
    print(f"  平均盈利: {metrics['avg_win']:.4%}")
    print(f"  平均亏损: {metrics['avg_loss']:.4%}")
    print(f"  单笔期望收益: {metrics['expectancy']:.4%}")
    print(f"  最佳交易: {metrics['best_trade']:.4%}")
    print(f"  最差交易: {metrics['worst_trade']:.4%}")
    print("-" * 60)
    print("【风险指标】")
    print(f"  夏普比率: {metrics['sharpe_ratio']:.3f}")
    print(f"  最大回撤: {metrics['max_drawdown']:.2%}")
    print(f"  最大回撤持续期 (bars): {metrics['max_drawdown_duration']}")
    print(f"  卡尔玛比率: {metrics['calmar_ratio']:.3f}")
    print(f"  年化波动率: {metrics['annualized_volatility']:.2%}")
    print("-" * 60)
    print("【费用统计】")
    print(f"  总手续费: {metrics['total_fees']:.6f}")
    print(f"  平均每笔手续费: {metrics['avg_fee_per_trade']:.6f}")
    print(f"  费率设置: {metrics['fee_rate']:.6f}")
    print("=" * 60)


def save_metrics_to_file(metrics, trades_df, output_dir, factor_name, fw, ema_w):
    """保存指标到文件"""
    # 创建输出目录
    save_dir = output_dir / factor_name / f"fw{fw}_ema{ema_w}" / "metrics"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 保存指标摘要为JSON
    metrics_serializable = {k: (str(v) if isinstance(v, pd.Timestamp) else v) 
                           for k, v in metrics.items()}
    with open(save_dir / "metrics_summary.json", 'w') as f:
        json.dump(metrics_serializable, f, indent=4, default=str)
    
    # 保存交易明细为CSV
    if not trades_df.empty:
        trades_df.to_csv(save_dir / "trades_detail.csv")
    
    # 保存文本报告
    with open(save_dir / "report.txt", 'w') as f:
        # 重定向print输出到文件
        import sys
        original_stdout = sys.stdout
        sys.stdout = f
        print_metrics_summary(metrics)
        sys.stdout = original_stdout


def plot_enhanced_results(bars, metrics, trades_df, factor_name, fw, ema_w, output_dir):
    """绘制增强版结果图表"""
    
    # 创建3行图表
    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        subplot_titles=('累计收益对比', '策略 vs 基准（滚动）', 
                       '多空累计收益（不含费用）', '回撤曲线',
                       '交易分布', '月度收益热力图'),
        vertical_spacing=0.08,
        horizontal_spacing=0.1
    )
    
    # 1. 累计收益对比（含费用）
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['benchmark_cumulative'], 
                  mode='lines', name='基准', line=dict(color='#1f77b4', width=1)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars[f'strategy_cumulative_{factor_name}'], 
                  mode='lines', name='策略（净）', line=dict(color='#2ecc71', width=2)),
        row=1, col=1
    )
    
    # 2. 滚动夏普比率（使用60期窗口）
    rolling_sharpe = bars[f'strategy_returns_{factor_name}'].rolling(60).mean() / \
                     bars[f'strategy_returns_{factor_name}'].rolling(60).std() * np.sqrt(60)
    fig.add_trace(
        go.Scatter(x=bars.index, y=rolling_sharpe, 
                  mode='lines', name='滚动夏普(60期)', 
                  line=dict(color='#ff7f0e', width=1)),
        row=1, col=2
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)
    
    # 3. 多空累计收益（不含费用）
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['cumulative_long_raw'], 
                  mode='lines', name='多头收益（原始）', line=dict(color='#d62728', width=1)),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['cumulative_short_raw'], 
                  mode='lines', name='空头收益（原始）', line=dict(color='#9467bd', width=1)),
        row=2, col=1
    )
    
    # 4. 回撤曲线
    rolling_max = bars[f'strategy_cumulative_{factor_name}'].expanding().max()
    drawdown = (bars[f'strategy_cumulative_{factor_name}'] - rolling_max) / rolling_max
    fig.add_trace(
        go.Scatter(x=bars.index, y=drawdown, fill='tozeroy',
                  mode='lines', name='回撤', line=dict(color='#e377c2', width=1)),
        row=2, col=2
    )
    fig.add_hline(y=metrics['max_drawdown'], line_dash="dash", 
                  line_color="red", opacity=0.5, row=2, col=2)
    
    # 5. 交易分布散点图
    if not trades_df.empty:
        # 多头交易
        long_trades = trades_df[trades_df['direction'] == 1]
        short_trades = trades_df[trades_df['direction'] == -1]
        
        fig.add_trace(
            go.Scatter(x=long_trades['entry_time'] if not long_trades.empty else [],
                      y=long_trades['return_raw'] if not long_trades.empty else [],
                      mode='markers', name='多头交易',
                      marker=dict(color='green', size=8, symbol='triangle-up'),
                      text=[f"收益: {r:.2%}" for r in long_trades['return_raw']] if not long_trades.empty else []),
            row=3, col=1
        )
        fig.add_trace(
            go.Scatter(x=short_trades['entry_time'] if not short_trades.empty else [],
                      y=short_trades['return_raw'] if not short_trades.empty else [],
                      mode='markers', name='空头交易',
                      marker=dict(color='red', size=8, symbol='triangle-down'),
                      text=[f"收益: {r:.2%}" for r in short_trades['return_raw']] if not short_trades.empty else []),
            row=3, col=1
        )
    
    # 6. 收益分布直方图
    fig.add_trace(
        go.Histogram(x=bars[f'strategy_returns_{factor_name}'], nbinsx=50,
                    name='收益分布', marker_color='#7f7f7f'),
        row=3, col=2
    )
    fig.add_vline(x=0, line_dash="dash", line_color="red", row=3, col=2)
    
    # 更新布局
    fig.update_layout(
        height=1200,
        showlegend=True,
        title_text=f"策略分析报告 - {factor_name} (fw={fw}, ema={ema_w})",
        template="plotly_white",
        hovermode='x unified'
    )
    
    # 更新坐标轴标签
    fig.update_xaxes(title_text="时间", row=3, col=1)
    fig.update_xaxes(title_text="时间", row=3, col=2)
    fig.update_yaxes(title_text="累计收益", row=1, col=1)
    fig.update_yaxes(title_text="夏普比率", row=1, col=2)
    fig.update_yaxes(title_text="累计收益", row=2, col=1)
    fig.update_yaxes(title_text="回撤", row=2, col=2, tickformat='.1%')
    fig.update_yaxes(title_text="交易收益", row=3, col=1, tickformat='.1%')
    fig.update_yaxes(title_text="频次", row=3, col=2)
    
    # 保存
    html_path = output_dir / factor_name / f"fw{fw}_ema{ema_w}" / "enhanced_analysis.html"
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    
    return fig

## SOTA

In [ ]:
def calculate_delta_threshold(delta_series, window=100, method='mad', k=1, vol_adjusted=False, vol_series=None, local_window=10):
    """
    计算delta的自适应阈值
    
    Parameters:
    -----------
    delta_series : pd.Series
        delta值序列
    window : int
        滚动窗口
    method : str
        'mad' - 基于绝对中位差（推荐，稳健）
        'std' - 基于标准差
        'quantile' - 基于分位数
    k : float
        阈值倍数
    """
    if method == 'mad':
        # 滚动中位数
        rolling_median = delta_series.rolling(window).median()
        # 绝对偏差
        abs_dev = (delta_series - rolling_median).abs()
        # 滚动MAD
        rolling_mad = abs_dev.rolling(window).median()
        threshold = rolling_mad * k
        
    elif method == 'std':
        rolling_std = delta_series.rolling(window).std()
        threshold = rolling_std * k
        
    elif method == 'quantile':
        threshold = delta_series.rolling(window).quantile(0.75)

    if vol_adjusted:
        local_vol_series = vol_series.rolling(local_window).mean()
        avg_vol = local_vol_series.rolling(window).mean()
        threshold = threshold * np.sqrt(local_vol_series / avg_vol)

    # TODO: 增加 local std
    
    return threshold.fillna(method='bfill').fillna(method='ffill')

# 修改你的原始函数，集成新功能
def vwap_dist_sum_v41_strategy_enhanced(bars, fw1, ema_w1, fw2, ema_w2, fill_rate=0.7, save_metrics=True, plot_enhanced=True):
    """
    增强版策略函数，包含全面绩效指标记录
    """
    factor_name = 'vwap_dist_sum_imbalance_v41'
    
    # 原有的因子计算代码保持不变
    bars[f'buy_vwap_dist_sum_fw{fw1}'] = bars['buy_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'sell_vwap_dist_sum_fw{fw1}'] = bars['sell_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'{factor_name}_fw{fw1}'] = (bars[f'buy_vwap_dist_sum_fw{fw1}'] - bars[f'sell_vwap_dist_sum_fw{fw1}']) / (bars[f'buy_vwap_dist_sum_fw{fw1}'] + bars[f'sell_vwap_dist_sum_fw{fw1}'] + 1e-10)
    bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'] = bars[f'{factor_name}_fw{fw1}'].ewm(span=ema_w1, min_periods=1).mean()

    bars[f'buy_vwap_dist_sum_fw{fw2}'] = bars['buy_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'sell_vwap_dist_sum_fw{fw2}'] = bars['sell_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'{factor_name}_fw{fw2}'] = (bars[f'buy_vwap_dist_sum_fw{fw2}'] - bars[f'sell_vwap_dist_sum_fw{fw2}']) / (bars[f'buy_vwap_dist_sum_fw{fw2}'] + bars[f'sell_vwap_dist_sum_fw{fw2}'] + 1e-10)
    bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'] = bars[f'{factor_name}_fw{fw2}'].ewm(span=ema_w2, min_periods=1).mean()

    bars[f'delta1'] = bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'].diff()
    bars[f'delta2'] = bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'].diff()

    bars['delta_threshold1'] = calculate_delta_threshold(
        bars['delta1'], 
        window=100, 
        method='mad',  # 推荐使用MAD，对异常值稳健
        k=1.5,         #1.5倍MAD
        vol_adjusted=True, vol_series=bars['volume'], local_window=10
    )
    bars['delta_threshold2'] = calculate_delta_threshold(
        bars['delta2'], 
        window=100, 
        method='mad',  # 推荐使用MAD，对异常值稳健
        k=1,          #1倍MAD
        vol_adjusted=True, vol_series=bars['volume'], local_window=10
    )

    bars['delta_sign_raw1'] = np.sign(bars['delta1'])
    bars['delta_sign1'] = bars['delta_sign_raw1'].replace(0, method='ffill')
    bars['delta_sign_raw2'] = np.sign(bars['delta2'])
    bars['delta_sign2'] = bars['delta_sign_raw2'].replace(0, method='ffill')

    bars['reversal_neg'] = (bars['delta_sign2'].shift(1) == 1) & (bars['delta_sign2'] == -1)
    bars['reversal_pos'] = (bars['delta_sign1'].shift(1) == -1) & (bars['delta_sign1'] == 1)

    filter_delta1 = bars['delta1'].abs() > bars['delta_threshold1']
    filter_delta2 = bars['delta2'].abs() > bars['delta_threshold2']

    bars[f'short_entry_{factor_name}'] = bars['reversal_neg'] & filter_delta2
    bars[f'long_entry_{factor_name}'] = bars['reversal_pos'] & filter_delta1

    bars[f'position_{factor_name}'] = np.nan
    bars.loc[bars[f'short_entry_{factor_name}'], f'position_{factor_name}'] = -1
    bars.loc[bars[f'long_entry_{factor_name}'], f'position_{factor_name}'] = 1
    bars[f'position_{factor_name}'].ffill(inplace=True)
    # 未开仓前用0填充
    bars[f'position_{factor_name}'] = bars[f'position_{factor_name}'].fillna(0)

    # # 画 indicators 图
    # draw_price_factor_graph_w_indicators(bars.copy(), factor_name, fw, ema_w)

    # 计算 pnl (考虑交易成本）
    fee_rate = (2 * fill_rate + 5 * (1 - fill_rate)) * 1e-4

    bars['returns'] = (bars['close'] - bars['open']) / bars['open'] # i-th bar returns
    bars[f'strategy_returns_{factor_name}'] = bars[f'position_{factor_name}'].shift(1) * bars['returns'] # i-th bar strategy returns

    # 考虑交易费用
    bars['position_change'] = bars[f'position_{factor_name}'].diff().fillna(0)
    bars[f'strategy_returns_{factor_name}'] = bars[f'strategy_returns_{factor_name}'] - fee_rate * bars['position_change'].abs()

    bars['benchmark_cumulative'] = (1 + bars['returns']).cumprod()
    bars[f'strategy_cumulative_{factor_name}'] = (1 + bars[f'strategy_returns_{factor_name}']).cumprod()

    # 新增：计算全面指标
    metrics, trades_df, bars_with_metrics = calculate_trading_metrics(
        bars, f'position_{factor_name}', f'strategy_returns_{factor_name}', 
        fee_rate, factor_name, fw1, ema_w1
    )
    
    # 打印指标汇总
    print_metrics_summary(metrics)
    
    # 保存指标到文件
    output_dir = Path("/Volumes/Lexar/mean_reversion_data/backtest_results")
    if save_metrics:
        save_metrics_to_file(metrics, trades_df, output_dir, factor_name, fw1, ema_w1)
    
    # 绘制增强版图表
    if plot_enhanced:
        plot_enhanced_results(bars_with_metrics, metrics, trades_df, factor_name, fw1, ema_w1, output_dir)
    
    return bars_with_metrics, metrics, trades_df

In [ ]:
vwap_dist_sum_v41_strategy_enhanced(bars, 8, 10, 10, 10)

## 对比使用 Volume 进行聚合

先利用已有数据检查volume与信号std的关系
=> 确定根据 volume build bar 的有效性

检查 volume 设置为多少合适

## 对比 sqrt(volume) 调整后的信号

但根据 volume 聚合应该是从根本上解决问题

## 基于 +DI, -DI 对开仓阈值进行调整

## 使用 MAE, MFE 进行止损优化

## 尝试 Grid Trading

## 结合 Orderbook 判断 Liquidity 条件是否具备开仓条件

## 提高杠杆

## 优化仓位配置